# NB_10 — Stage 10: Arabic Text Normalization Impact

**Purpose:** Measure how much of the CER/WER reported in NB_06, NB_08, and NB_09 is attributable
to orthographic surface variation rather than genuine transcription errors.

Arabic text has several sources of surface variation that are linguistically meaningless but
cause CER to spike artificially when reference and hypothesis use different but equivalent forms:

| Step | What it does | Example |
|---|---|---|
| Alef normalization | Map أ، إ، آ → ا | إسلام → اسلام |
| Taa marbuta | Map ة → ه | مدرسة → مدرسه |
| Diacritics removal | Strip harakat (fatha, kasra, damma, etc.) | كَتَبَ → كتب |
| Kashida removal | Strip elongation character ـ | الـكتاب → الكتاب |
| Punctuation | Strip Arabic punctuation marks | النص. → النص |

Applying these steps to both reference and hypothesis before scoring gives a
**normalized CER/WER** that reflects genuine reading errors only.

**No GPU required.** No retraining. Runs entirely on saved eval result JSONs.

**Prerequisites:**
- NB_06 completed: `logs/run-1/eval_results_checkpoint-1120.json` (with `per_sample`)
- NB_09 completed: `logs/run-1/eval_results_zero_shot.json` (with `per_sample`)
- NB_08 completed: `logs/stage7/easyocr_results.json`
- `data/eval/eval.jsonl` (for rebuilding EasyOCR pairs over all 280 samples)

## Step 10.1 — Mount Drive and set paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project'

# Input files — all produced by earlier notebooks
# NB_05/06 used RUN_NAME='run-1' (renamed on Drive after training)
QWEN_EVAL_FILE     = f'{PROJECT_ROOT}/logs/run-1/eval_results_checkpoint-1120.json'
ZEROSHOT_EVAL_FILE = f'{PROJECT_ROOT}/logs/run-1/eval_results_zero_shot.json'
EASYOCR_FILE       = f'{PROJECT_ROOT}/logs/stage7/easyocr_results.json'
EVAL_JSONL         = f'{PROJECT_ROOT}/data/eval/eval.jsonl'

# Output
RESULTS_DIR = f'{PROJECT_ROOT}/logs/stage10'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Verify all input files exist before proceeding
missing = []
for label, path in [
    ('Qwen eval results',     QWEN_EVAL_FILE),
    ('Zero-shot eval results', ZEROSHOT_EVAL_FILE),
    ('EasyOCR results',        EASYOCR_FILE),
    ('Eval JSONL',             EVAL_JSONL),
]:
    exists = os.path.exists(path)
    print(f'  {"✓" if exists else "✗"} {label}: {path}')
    if not exists:
        missing.append(label)

if missing:
    raise FileNotFoundError(
        f'Missing files: {missing}\n'
        f'Run NB_06, NB_08, NB_09 before this notebook.'
    )
print('\nAll input files found.')

Mounted at /content/drive
  ✓ Qwen eval results: /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project/logs/run-1/eval_results_checkpoint-1120.json
  ✓ Zero-shot eval results: /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project/logs/run-1/eval_results_zero_shot.json
  ✓ EasyOCR results: /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project/logs/stage7/easyocr_results.json
  ✓ Eval JSONL: /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project/data/eval/eval.jsonl

All input files found.


## Step 10.2 — Install dependencies

`jiwer` is the only external dependency. `camel-tools` is NOT used here — the
normalization is implemented directly using Unicode character ranges so there are
no version conflicts to manage.

In [3]:
import subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'jiwer', '--quiet'],
    check=True
)

import jiwer
import importlib.metadata
print(f'jiwer version: {importlib.metadata.version("jiwer")}')
print('Dependencies installed.')

jiwer version: 4.0.0
Dependencies installed.


## Step 10.3 — Define the Arabic normalization function

The five normalization steps are applied in a fixed order. Diacritics are stripped
first because some diacritics attach to alef variants and should be removed before
alef normalization collapses those variants.

All character ranges are from the Unicode Arabic block (U+0600–U+06FF) and the
Arabic Presentation Forms blocks (U+FB50–U+FDFF, U+FE70–U+FEFF).

In [5]:
import re
import unicodedata


def normalize_arabic(text: str) -> str:
    """
    Apply five standard Arabic normalization steps in order:
    1. Diacritics (harakat) removal
    2. Alef normalization  (أ إ آ أ ٱ → ا)
    3. Taa marbuta         (ة → ه)
    4. Kashida removal     (ـ)
    5. Punctuation removal (Arabic punctuation + common Latin punctuation)
    """
    if not text or not isinstance(text, str):
        return ''

    # ── 1. Diacritics (harakat) ───────────────────────────────────────────
    # Arabic diacritical marks: U+064B (fathatan) through U+065F
    # Also U+0670 (superscript alef) and U+0640 retained for kashida step
    DIACRITICS = re.compile(r'[\u064B-\u065F\u0670]')
    text = DIACRITICS.sub('', text)

    # ── 2. Alef normalization ─────────────────────────────────────────────
    # أ (U+0623), إ (U+0625), آ (U+0622), أ (U+0623), ٱ (U+0671) → ا (U+0627)
    ALEF_VARIANTS = re.compile(r'[\u0622\u0623\u0625\u0671]')
    text = ALEF_VARIANTS.sub('\u0627', text)  # → bare alef ا

    # ── 3. Taa marbuta normalization ──────────────────────────────────────
    # ة (U+0629) → ه (U+0647)
    text = text.replace('\u0629', '\u0647')

    # ── 4. Kashida (tatweel) removal ──────────────────────────────────────
    # ـ (U+0640)
    text = text.replace('\u0640', '')

    # ── 5. Punctuation removal ────────────────────────────────────────────
    # Arabic punctuation: ، ؛ ؟ ۔ and common Latin: . , ; : ! ? ( ) [ ] - "
    PUNCTUATION = re.compile(
        r'[\u060C\u061B\u061F\u06D4'   # Arabic: comma, semicolon, question, full stop
        r'\u0021-\u002F'               # ! " # $ % & ' ( ) * + , - . /
        r'\u003A-\u0040'               # : ; < = > ? @
        r'\u005B-\u0060'               # [ \ ] ^ _ `
        r'\u007B-\u007E]'              # { | } ~
    )
    text = PUNCTUATION.sub('', text)

    # Collapse any resulting multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [6]:
# ── Sanity checks ─────────────────────────────────────────────────────────
tests = [
    ('إسلام',       'اسلام',    'Alef normalization'),
    ('مدرسة',       'مدرسه',    'Taa marbuta'),
    ('كَتَبَ',      'كتب',      'Diacritics removal'),
    ('الـكتاب',     'الكتاب',   'Kashida removal'),
    ('النص.',       'النص',     'Punctuation removal'),
    ('آمَنَّا بالله', 'امنا بالله', 'Combined'),
]

print('Normalization sanity checks:')
all_pass = True
for original, expected, label in tests:
    result = normalize_arabic(original)
    status = '✓' if result == expected else '✗'
    if result != expected:
        all_pass = False
    print(f'  {status} {label}:')
    print(f'       Input   : {original}')
    print(f'       Output  : {result}')
    print(f'       Expected: {expected}')
    print(f'       Match   : {"yes" if result == expected else "NO"}')

if all_pass:
    print('\nAll checks passed.')
else:
    print('\nSome checks failed — review normalization rules above.')

Normalization sanity checks:
  ✓ Alef normalization:
       Input   : إسلام
       Output  : اسلام
       Expected: اسلام
       Match   : yes
  ✓ Taa marbuta:
       Input   : مدرسة
       Output  : مدرسه
       Expected: مدرسه
       Match   : yes
  ✓ Diacritics removal:
       Input   : كَتَبَ
       Output  : كتب
       Expected: كتب
       Match   : yes
  ✓ Kashida removal:
       Input   : الـكتاب
       Output  : الكتاب
       Expected: الكتاب
       Match   : yes
  ✓ Punctuation removal:
       Input   : النص.
       Output  : النص
       Expected: النص
       Match   : yes
  ✓ Combined:
       Input   : آمَنَّا بالله
       Output  : امنا بالله
       Expected: امنا بالله
       Match   : yes

All checks passed.


## Step 10.4 — Load reference/hypothesis pairs for all three models

**Qwen fine-tuned and zero-shot:** loaded from `per_sample` arrays in their respective
eval result JSONs — each entry has `reference` and `hypothesis` fields covering all 280 samples.

**EasyOCR:** the `easyocr_results.json` only stores the first 10 `sample_outputs`.
For the full 280-sample comparison, we rebuild reference strings from `eval.jsonl`
and re-run EasyOCR inference inline. If you want to skip the re-inference, the
notebook will fall back to the 10 stored samples with a warning.

In [8]:
import json

In [9]:
# ── Qwen fine-tuned ───────────────────────────────────────────────────────
with open(QWEN_EVAL_FILE) as f:
    qwen_data = json.load(f)

if 'per_sample' not in qwen_data:
    raise KeyError(
        'per_sample not found in Qwen eval results.\n'
        'Make sure NB_06 saved the full results including per_sample.\n'
        f'Keys found: {list(qwen_data.keys())}'
    )

qwen_pairs = [
    (s['reference'], s['hypothesis'])
    for s in qwen_data['per_sample']
    if s.get('reference', '').strip()
]
print(f'Qwen fine-tuned pairs loaded: {len(qwen_pairs)}')

Qwen fine-tuned pairs loaded: 280


In [15]:
# ── Zero-shot Qwen ────────────────────────────────────────────────────────
with open(ZEROSHOT_EVAL_FILE) as f:
    zs_data = json.load(f)

if 'per_sample' not in zs_data:
    raise KeyError(
        'per_sample not found in zero-shot eval results.\n'
        f'Keys found: {list(zs_data.keys())}'
    )

zs_pairs = [
    (s['reference'], s['prediction'])
    for s in zs_data['per_sample']
    if s.get('reference', '').strip()
]
print(f'Zero-shot Qwen pairs loaded:  {len(zs_pairs)}')

Zero-shot Qwen pairs loaded:  280


In [16]:
# ── EasyOCR ───────────────────────────────────────────────────────────────
# easyocr_results.json only stores 10 sample_outputs.
# We load GPT references from eval.jsonl and pair with stored predictions.
# For the normalization analysis 10 samples are sufficient to illustrate
# the effect; the aggregate CER/WER uses the stored summary metrics.
with open(EASYOCR_FILE) as f:
    easyocr_data = json.load(f)

easyocr_pairs = [
    (s['reference'], s['predicted'])
    for s in easyocr_data.get('sample_outputs', [])
    if s.get('reference', '').strip()
]
print(f'EasyOCR pairs loaded: {len(easyocr_pairs)} (first 10 only — sufficient for examples)')

EasyOCR pairs loaded: 10 (first 10 only — sufficient for examples)


In [17]:
# ── GPT references (all 280, from eval.jsonl) ────────────────────────────
# Used as a standalone source for normalization statistics on the reference side
gpt_refs = []
with open(EVAL_JSONL) as f:
    for line in f:
        sample = json.loads(line)
        for msg in sample['messages']:
            if msg['role'] == 'assistant':
                try:
                    parsed = json.loads(msg['content'])
                    text = parsed.get('transcription', '').strip()
                    if text:
                        gpt_refs.append(text)
                except Exception:
                    pass
print(f'GPT reference strings loaded: {len(gpt_refs)}')

GPT reference strings loaded: 280


## Step 10.5 — Compute before/after normalization metrics

For each model we compute CER and WER twice:
- **Raw:** on the original strings as produced by the model
- **Normalized:** after applying all five normalization steps to both reference and hypothesis

The reduction tells us how much of the error rate was orthographic noise.

In [18]:
from jiwer import cer, wer


def compute_metrics(pairs, label):
    """
    Compute raw and normalized CER/WER for a list of (reference, hypothesis) pairs.
    Returns a dict with all four values plus reduction percentages.
    """
    refs_raw  = [r for r, h in pairs]
    hyps_raw  = [h for r, h in pairs]
    refs_norm = [normalize_arabic(r) for r in refs_raw]
    hyps_norm = [normalize_arabic(h) for h in hyps_raw]

    # Filter out empty references after normalization
    raw_pairs  = [(r, h) for r, h in zip(refs_raw,  hyps_raw)  if r.strip()]
    norm_pairs = [(r, h) for r, h in zip(refs_norm, hyps_norm) if r.strip()]

    raw_refs,  raw_hyps  = zip(*raw_pairs)  if raw_pairs  else ([], [])
    norm_refs, norm_hyps = zip(*norm_pairs) if norm_pairs else ([], [])

    raw_cer  = round(cer(list(raw_refs),  list(raw_hyps)),  4) if raw_refs  else 0.0
    raw_wer  = round(wer(list(raw_refs),  list(raw_hyps)),  4) if raw_refs  else 0.0
    norm_cer = round(cer(list(norm_refs), list(norm_hyps)), 4) if norm_refs else 0.0
    norm_wer = round(wer(list(norm_refs), list(norm_hyps)), 4) if norm_refs else 0.0

    cer_reduction = round((raw_cer  - norm_cer)  / raw_cer  * 100, 1) if raw_cer  > 0 else 0.0
    wer_reduction = round((raw_wer  - norm_wer)  / raw_wer  * 100, 1) if raw_wer  > 0 else 0.0

    result = {
        'label':         label,
        'n_samples':     len(raw_pairs),
        'raw_cer':       raw_cer,
        'raw_wer':       raw_wer,
        'norm_cer':      norm_cer,
        'norm_wer':      norm_wer,
        'cer_reduction': cer_reduction,
        'wer_reduction': wer_reduction,
    }

    print(f'\n=== {label} (n={len(raw_pairs)}) ===')
    print(f'  Raw  CER: {raw_cer:.4f}   WER: {raw_wer:.4f}')
    print(f'  Norm CER: {norm_cer:.4f}   WER: {norm_wer:.4f}')
    print(f'  CER reduction: {cer_reduction:.1f}%   WER reduction: {wer_reduction:.1f}%')

    return result, list(zip(refs_raw, hyps_raw, refs_norm, hyps_norm))


qwen_result,    qwen_detail    = compute_metrics(qwen_pairs,    'Qwen2.5-VL-7B + LoRA (checkpoint-1120)')
zs_result,      zs_detail      = compute_metrics(zs_pairs,      'Qwen2.5-VL-7B zero-shot')
easyocr_result, easyocr_detail = compute_metrics(easyocr_pairs, 'EasyOCR (Arabic, 10 samples)')


=== Qwen2.5-VL-7B + LoRA (checkpoint-1120) (n=280) ===
  Raw  CER: 0.3283   WER: 0.6554
  Norm CER: 0.3043   WER: 0.6160
  CER reduction: 7.3%   WER reduction: 6.0%

=== Qwen2.5-VL-7B zero-shot (n=280) ===
  Raw  CER: 0.4442   WER: 0.7549
  Norm CER: 0.3506   WER: 0.6931
  CER reduction: 21.1%   WER reduction: 8.2%

=== EasyOCR (Arabic, 10 samples) (n=10) ===
  Raw  CER: 0.4377   WER: 1.0388
  Norm CER: 0.4007   WER: 0.9608
  CER reduction: 8.5%   WER reduction: 7.5%


## Step 10.6 — Summary table

Print the complete before/after comparison across all three models.
This table goes directly into Section 5.1 of the paper.

In [19]:
results = [qwen_result, zs_result, easyocr_result]

print('=' * 80)
print('NORMALIZATION IMPACT SUMMARY')
print('=' * 80)
print(f'{"Model":<42} {"Raw CER":>8} {"Norm CER":>9} {"CER↓%":>7} {"Raw WER":>8} {"Norm WER":>9} {"WER↓%":>7}')
print('-' * 80)
for r in results:
    print(
        f'{r["label"]:<42} '
        f'{r["raw_cer"]:>8.4f} '
        f'{r["norm_cer"]:>9.4f} '
        f'{r["cer_reduction"]:>6.1f}% '
        f'{r["raw_wer"]:>8.4f} '
        f'{r["norm_wer"]:>9.4f} '
        f'{r["wer_reduction"]:>6.1f}%'
    )
print('=' * 80)
print('Note: EasyOCR metrics computed on first 10 samples only.')
print('      Qwen and zero-shot metrics computed on all 280 eval samples.')

NORMALIZATION IMPACT SUMMARY
Model                                       Raw CER  Norm CER   CER↓%  Raw WER  Norm WER   WER↓%
--------------------------------------------------------------------------------
Qwen2.5-VL-7B + LoRA (checkpoint-1120)       0.3283    0.3043    7.3%   0.6554    0.6160    6.0%
Qwen2.5-VL-7B zero-shot                      0.4442    0.3506   21.1%   0.7549    0.6931    8.2%
EasyOCR (Arabic, 10 samples)                 0.4377    0.4007    8.5%   1.0388    0.9608    7.5%
Note: EasyOCR metrics computed on first 10 samples only.
      Qwen and zero-shot metrics computed on all 280 eval samples.


## Step 10.7 — Per-step normalization breakdown

Apply each normalization step individually to the Qwen fine-tuned pairs to show
which step contributes most to the CER reduction. This gives a more granular
picture for the discussion section of the paper.

In [20]:
import re

def step_diacritics(text):
    return re.compile(r'[\u064B-\u065F\u0670]').sub('', text)

def step_alef(text):
    return re.compile(r'[\u0622\u0623\u0625\u0671]').sub('\u0627', text)

def step_taa(text):
    return text.replace('\u0629', '\u0647')

def step_kashida(text):
    return text.replace('\u0640', '')

def step_punct(text):
    PUNCTUATION = re.compile(
        r'[\u060C\u061B\u061F\u06D4'
        r'\u0021-\u002F\u003A-\u0040\u005B-\u0060\u007B-\u007E]'
    )
    return re.sub(r'\s+', ' ', PUNCTUATION.sub('', text)).strip()


steps = [
    ('Diacritics removal',      step_diacritics),
    ('Alef normalization',      step_alef),
    ('Taa marbuta',             step_taa),
    ('Kashida removal',         step_kashida),
    ('Punctuation removal',     step_punct),
    ('All steps combined',      normalize_arabic),
]

refs_raw = [r for r, h in qwen_pairs]
hyps_raw = [h for r, h in qwen_pairs]

# Baseline
baseline_cer = round(cer(refs_raw, hyps_raw), 4)
baseline_wer = round(wer(refs_raw, hyps_raw), 4)

print('Per-step normalization breakdown (Qwen fine-tuned, 280 samples)')
print(f'{"Step":<28} {"CER":>8} {"CER↓":>7} {"WER":>8} {"WER↓":>7}')
print('-' * 60)
print(f'{"Baseline (no normalization)":<28} {baseline_cer:>8.4f} {"":>7} {baseline_wer:>8.4f} {"":>7}')

for step_name, step_fn in steps:
    refs_norm = [step_fn(r) for r in refs_raw]
    hyps_norm = [step_fn(h) for h in hyps_raw]

    # Filter empty
    pairs = [(r, h) for r, h in zip(refs_norm, hyps_norm) if r.strip()]
    rs, hs = zip(*pairs) if pairs else ([], [])

    step_cer = round(cer(list(rs), list(hs)), 4) if rs else 0.0
    step_wer = round(wer(list(rs), list(hs)), 4) if rs else 0.0

    cer_delta = round(baseline_cer - step_cer, 4)
    wer_delta = round(baseline_wer - step_wer, 4)

    print(f'{step_name:<28} {step_cer:>8.4f} {cer_delta:>+7.4f} {step_wer:>8.4f} {wer_delta:>+7.4f}')

Per-step normalization breakdown (Qwen fine-tuned, 280 samples)
Step                              CER    CER↓      WER    WER↓
------------------------------------------------------------
Baseline (no normalization)    0.3283           0.6554        
Diacritics removal             0.3227 +0.0056   0.6495 +0.0059
Alef normalization             0.3220 +0.0063   0.6492 +0.0062
Taa marbuta                    0.3248 +0.0035   0.6512 +0.0042
Kashida removal                0.3283 +0.0000   0.6554 +0.0000
Punctuation removal            0.3204 +0.0079   0.6335 +0.0219
All steps combined             0.3043 +0.0240   0.6160 +0.0394


## Step 10.8 — Detailed error examples

Find the 5 Qwen samples where normalization produces the largest CER reduction.
These are the clearest cases where orthographic variation was being penalized
as error — the normalized versions show what genuine transcription quality looks like.

In [21]:
from jiwer import cer as jiwer_cer

# Compute per-sample CER before and after normalization
per_sample_gains = []

for ref_raw, hyp_raw, ref_norm, hyp_norm in qwen_detail:
    if not ref_raw.strip() or not ref_norm.strip():
        continue

    try:
        raw_s  = float(jiwer_cer(ref_raw,  hyp_raw))
        norm_s = float(jiwer_cer(ref_norm, hyp_norm))
    except Exception:
        continue

    reduction = raw_s - norm_s
    per_sample_gains.append({
        'ref_raw':   ref_raw,
        'hyp_raw':   hyp_raw,
        'ref_norm':  ref_norm,
        'hyp_norm':  hyp_norm,
        'cer_raw':   round(raw_s,  4),
        'cer_norm':  round(norm_s, 4),
        'reduction': round(reduction, 4),
    })

# Sort by largest reduction
per_sample_gains.sort(key=lambda x: -x['reduction'])

print('=== Top 5 Samples: Largest CER Reduction from Normalization ===')
for rank, s in enumerate(per_sample_gains[:5], 1):
    print(f'\n[{rank}] CER: {s["cer_raw"]:.4f} → {s["cer_norm"]:.4f}  (reduction: {s["reduction"]:.4f})')
    print(f'  REF  (raw) : {s["ref_raw"]}')
    print(f'  HYP  (raw) : {s["hyp_raw"]}')
    print(f'  REF  (norm): {s["ref_norm"]}')
    print(f'  HYP  (norm): {s["hyp_norm"]}')

print('\n=== Top 5 Samples: Smallest CER Reduction (normalization had little effect) ===')
for rank, s in enumerate(sorted(per_sample_gains, key=lambda x: x['reduction'])[:5], 1):
    print(f'\n[{rank}] CER: {s["cer_raw"]:.4f} → {s["cer_norm"]:.4f}  (reduction: {s["reduction"]:.4f})')
    print(f'  REF (raw) : {s["ref_raw"]}')
    print(f'  HYP (raw) : {s["hyp_raw"]}')

=== Top 5 Samples: Largest CER Reduction from Normalization ===

[1] CER: 1.5000 → 0.5000  (reduction: 1.0000)
  REF  (raw) : تأشيرة
  HYP  (raw) : تُأَمِّنْهُ
  REF  (norm): تاشيره
  HYP  (norm): تامنه

[2] CER: 0.4000 → 0.0000  (reduction: 0.4000)
  REF  (raw) : جميلة
  HYP  (raw) : جميلة .
  REF  (norm): جميله
  HYP  (norm): جميله

[3] CER: 0.5234 → 0.2297  (reduction: 0.2937)
  REF  (raw) : وَقَدْ ذَكَرَ السُّعُودِيُّ أَنَّ تَوْرَةَ أَصْحَابِ الْفِلْسَفَةِ كَانَ يَوْمَ الأَحَدِ سَبْعَ عَشْرَةَ لَيْلَةً مِنْ رَمَضَانَ
  HYP  (raw) : وقد ذكر السعدي أن قوم أصحاب الفضل مكة، كان يوم الأحد لسبع عشرة ليلة حلت من العام
  REF  (norm): وقد ذكر السعودي ان توره اصحاب الفلسفه كان يوم الاحد سبع عشره ليله من رمضان
  HYP  (norm): وقد ذكر السعدي ان قوم اصحاب الفضل مكه كان يوم الاحد لسبع عشره ليله حلت من العام

[4] CER: 1.1250 → 0.8750  (reduction: 0.2500)
  REF  (raw) : من يعترف
  HYP  (raw) : من لكو لفته ،
  REF  (norm): من يعترف
  HYP  (norm): من لكو لفته

[5] CER: 0.4074 → 0.2745  (reduction: 0.

## Step 10.9 — Normalization statistics on the reference corpus

Characterize how much orthographic variation exists in the GPT reference
transcriptions themselves. This contextualizes why normalization matters:
if GPT uses diacritics inconsistently, any model that learns from it will
inherit that inconsistency.

In [22]:
import re

DIACRITICS = re.compile(r'[\u064B-\u065F\u0670]')
ALEF_VARIANTS = re.compile(r'[\u0622\u0623\u0625\u0671]')
KASHIDA = re.compile(r'\u0640')
TAA_MARBUTA = re.compile(r'\u0629')

total_chars         = sum(len(t) for t in gpt_refs)
diacritic_chars     = sum(len(DIACRITICS.findall(t))    for t in gpt_refs)
alef_variant_chars  = sum(len(ALEF_VARIANTS.findall(t)) for t in gpt_refs)
kashida_chars       = sum(len(KASHIDA.findall(t))       for t in gpt_refs)
taa_marbuta_chars   = sum(len(TAA_MARBUTA.findall(t))   for t in gpt_refs)
texts_with_diacritics   = sum(1 for t in gpt_refs if DIACRITICS.search(t))
texts_with_alef_variants = sum(1 for t in gpt_refs if ALEF_VARIANTS.search(t))

print('=== GPT Reference Corpus Normalization Statistics ===')
print(f'Total transcriptions : {len(gpt_refs)}')
print(f'Total characters     : {total_chars:,}')
print()
print(f'Diacritic chars      : {diacritic_chars:,}  ({diacritic_chars/total_chars*100:.2f}% of chars)')
print(f'Texts with diacritics: {texts_with_diacritics} / {len(gpt_refs)} transcriptions')
print()
print(f'Alef variants        : {alef_variant_chars:,}  ({alef_variant_chars/total_chars*100:.2f}% of chars)')
print(f'Texts with alef vars : {texts_with_alef_variants} / {len(gpt_refs)} transcriptions')
print()
print(f'Kashida chars        : {kashida_chars:,}  ({kashida_chars/total_chars*100:.2f}% of chars)')
print(f'Taa marbuta chars    : {taa_marbuta_chars:,}  ({taa_marbuta_chars/total_chars*100:.2f}% of chars)')

=== GPT Reference Corpus Normalization Statistics ===
Total transcriptions : 280
Total characters     : 16,809

Diacritic chars      : 100  (0.59% of chars)
Texts with diacritics: 21 / 280 transcriptions

Alef variants        : 391  (2.33% of chars)
Texts with alef vars : 193 / 280 transcriptions

Kashida chars        : 0  (0.00% of chars)
Taa marbuta chars    : 421  (2.50% of chars)


## Step 10.10 — Save all results

In [24]:
import json, os

output = {
    'normalization_steps': [
        'diacritics_removal',
        'alef_normalization',
        'taa_marbuta',
        'kashida_removal',
        'punctuation_removal',
    ],
    'models': [
        {
            'label':         r['label'],
            'n_samples':     r['n_samples'],
            'raw_cer':       r['raw_cer'],
            'norm_cer':      r['norm_cer'],
            'cer_reduction': r['cer_reduction'],
            'raw_wer':       r['raw_wer'],
            'norm_wer':      r['norm_wer'],
            'wer_reduction': r['wer_reduction'],
        }
        for r in results
    ],
    'top5_normalization_examples': [
        {
            'ref_raw':   s['ref_raw'],
            'hyp_raw':   s['hyp_raw'],
            'ref_norm':  s['ref_norm'],
            'hyp_norm':  s['hyp_norm'],
            'cer_raw':   s['cer_raw'],
            'cer_norm':  s['cer_norm'],
            'reduction': s['reduction'],
        }
        for s in per_sample_gains[:5]
    ],
    'reference_corpus_stats': {
        'total_transcriptions':    len(gpt_refs),
        'total_chars':             total_chars,
        'diacritic_chars':         diacritic_chars,
        'diacritic_pct':           round(diacritic_chars / total_chars * 100, 2),
        'texts_with_diacritics':   texts_with_diacritics,
        'alef_variant_chars':      alef_variant_chars,
        'alef_variant_pct':        round(alef_variant_chars / total_chars * 100, 2),
        'texts_with_alef_variants':texts_with_alef_variants,
        'kashida_chars':           kashida_chars,
        'taa_marbuta_chars':       taa_marbuta_chars,
    },
    'note': (
        'Normalization applied to both reference and hypothesis before scoring. '
        'CER/WER computed with jiwer. EasyOCR metrics based on 10 samples only. '
        'Qwen and zero-shot metrics based on all 280 eval samples.'
    ),
}

out_path = f'{RESULTS_DIR}/normalization_results.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f'Results saved to {out_path}')
print('\nNB_10 complete.')

Results saved to /content/drive/MyDrive/CUD files/NLP/NLP_Arabic_HTR_Project/logs/stage10/normalization_results.json

NB_10 complete.
